# NeuroGaze Model Training Notebook

This notebook provides training capabilities for the neurogaze attention monitoring system.

In [ ]:
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os
import sys

# Add parent directory to path
sys.path.append('..')
from engine.attention_engine import analyze_attention_frame

## Data Collection

In [ ]:
def collect_training_data(duration_minutes=5):
    """Collect training data from webcam"""
    cap = cv2.VideoCapture(0)
    data = []
    labels = []
    
    print("Data collection started. Press 'a' for alert, 'd' for drowsy, 'n' for distracted, 'q' to quit")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        # Analyze frame
        distraction_ratio, drowsiness_ratio, annotated_frame = analyze_attention_frame(frame)
        
        cv2.imshow('Training Data Collection', annotated_frame)
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('a'):  # Alert
            data.append([distraction_ratio, drowsiness_ratio])
            labels.append('alert')
            print(f"Alert sample collected: {len(data)}")
        elif key == ord('d'):  # Drowsy
            data.append([distraction_ratio, drowsiness_ratio])
            labels.append('drowsy')
            print(f"Drowsy sample collected: {len(data)}")
        elif key == ord('n'):  # Distracted
            data.append([distraction_ratio, drowsiness_ratio])
            labels.append('distracted')
            print(f"Distracted sample collected: {len(data)}")
        elif key == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    
    return np.array(data), np.array(labels)

# Collect data (uncomment to run)
# X, y = collect_training_data()
# np.save('training_features.npy', X)
# np.save('training_labels.npy', y)

## Load Training Data

In [ ]:
# Load collected data
try:
    X = np.load('training_features.npy')
    y = np.load('training_labels.npy')
    print(f"Loaded {len(X)} samples with {len(np.unique(y))} classes")
except FileNotFoundError:
    print("No training data found. Run data collection first.")
    # Create sample data for demonstration
    X = np.random.rand(100, 2) * 100
    y = np.random.choice(['alert', 'drowsy', 'distracted'], 100)

## Data Visualization

In [ ]:
# Visualize data distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.scatter(X[:, 0], X[:, 1], c=[{'alert': 0, 'drowsy': 1, 'distracted': 2}[label] for label in y])
plt.xlabel('Distraction Ratio')
plt.ylabel('Drowsiness Ratio')
plt.title('Feature Distribution')
plt.colorbar()

plt.subplot(1, 3, 2)
plt.hist(X[:, 0], bins=20, alpha=0.7)
plt.xlabel('Distraction Ratio')
plt.title('Distraction Distribution')

plt.subplot(1, 3, 3)
plt.hist(X[:, 1], bins=20, alpha=0.7)
plt.xlabel('Drowsiness Ratio')
plt.title('Drowsiness Distribution')

plt.tight_layout()
plt.show()

## Model Training

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## Threshold Optimization

In [ ]:
def optimize_thresholds(X, y):
    """Find optimal thresholds for attention detection"""
    thresholds = {
        'drowsy_threshold': None,
        'distracted_threshold': None
    }
    
    # Find drowsiness threshold
    drowsy_samples = X[y == 'drowsy']
    alert_samples = X[y == 'alert']
    
    if len(drowsy_samples) > 0 and len(alert_samples) > 0:
        drowsy_mean = np.mean(drowsy_samples[:, 1])  # drowsiness ratio
        alert_mean = np.mean(alert_samples[:, 1])
        thresholds['drowsy_threshold'] = (drowsy_mean + alert_mean) / 2
    
    # Find distraction threshold
    distracted_samples = X[y == 'distracted']
    
    if len(distracted_samples) > 0 and len(alert_samples) > 0:
        distracted_mean = np.mean(distracted_samples[:, 0])  # distraction ratio
        alert_mean = np.mean(alert_samples[:, 0])
        thresholds['distracted_threshold'] = (distracted_mean + alert_mean) / 2
    
    return thresholds

optimal_thresholds = optimize_thresholds(X, y)
print("Optimal Thresholds:")
print(f"Drowsy threshold: {optimal_thresholds['drowsy_threshold']:.2f}")
print(f"Distracted threshold: {optimal_thresholds['distracted_threshold']:.2f}")

## Save Model

In [ ]:
# Save trained model and thresholds
joblib.dump(clf, '../models/attention_classifier.pkl')
np.save('../models/optimal_thresholds.npy', optimal_thresholds)

print("Model and thresholds saved successfully!")

## Real-time Testing

In [ ]:
def test_model_realtime():
    """Test the trained model in real-time"""
    cap = cv2.VideoCapture(0)
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        # Analyze frame
        distraction_ratio, drowsiness_ratio, annotated_frame = analyze_attention_frame(frame)
        
        # Predict using trained model
        features = np.array([[distraction_ratio, drowsiness_ratio]])
        prediction = clf.predict(features)[0]
        confidence = np.max(clf.predict_proba(features))
        
        # Display prediction
        cv2.putText(annotated_frame, f'State: {prediction} ({confidence:.2f})', 
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(annotated_frame, f'Distraction: {distraction_ratio:.1f}', 
                   (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
        cv2.putText(annotated_frame, f'Drowsiness: {drowsiness_ratio:.1f}', 
                   (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
        
        cv2.imshow('Model Testing', annotated_frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()

# Uncomment to test
# test_model_realtime()